# 01b - Staff and Customer Zones

نوتبوك مستقل لا يغيّر `local_tracks.csv`. يقرأ التتبع المحلي وفيديوهات `Data/raw` و`Data/config/staff_zones.json`، ثم يكتب `track_roles.csv` و`staff_customer_summary.csv` وفيديوهات جديدة باسم `roles_<camera_id>.mp4`. إحداثيات staff zones هي **بكسل الكاميرا** وليست إحداثيات أرضية أو هوية عبر كاميرات.

In [ ]:
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'Data' / 'raw'
TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
VIDEOS_DIR = PROJECT_ROOT / 'Output' / 'videos'
LOCAL_TRACKS_PATH = TABLES_DIR / 'local_tracks.csv'
STAFF_ZONES_PATH = PROJECT_ROOT / 'Data' / 'config' / 'staff_zones.json'
EMPLOYEE_IDS_PATH = PROJECT_ROOT / 'Data' / 'config' / 'employee_local_ids.json'
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}

if not LOCAL_TRACKS_PATH.exists():
    raise FileNotFoundError('local_tracks.csv is missing. Run Notebook 01 first.')
if not STAFF_ZONES_PATH.exists():
    raise FileNotFoundError('staff_zones.json is missing. Create it with [] or run pick_staff_zone.py.')
local_tracks = pd.read_csv(LOCAL_TRACKS_PATH)
required_columns = {'camera_id', 'local_track_id', 'frame_index', 'timestamp_sec', 'foot_x', 'foot_y'}
missing_columns = sorted(required_columns.difference(local_tracks.columns))
if missing_columns:
    raise ValueError(f'local_tracks.csv is missing columns: {missing_columns}')
if local_tracks.empty:
    raise ValueError('local_tracks.csv is empty. Run Notebook 01 with readable video first.')
staff_zones = json.loads(STAFF_ZONES_PATH.read_text(encoding='utf-8'))
if not isinstance(staff_zones, list):
    raise ValueError('staff_zones.json must be a JSON list, like store_zones.json, with an added camera_id field.')
for zone in staff_zones:
    expected = {'camera_id', 'zone_id', 'label_ar', 'kind', 'polygon'}
    if not expected.issubset(zone):
        raise ValueError(f'Staff zone is missing keys: {sorted(expected.difference(zone))}')
    if len(zone['polygon']) < 3:
        raise ValueError(f"Staff zone {zone['zone_id']} needs at least three polygon points.")
employee_local_ids = json.loads(EMPLOYEE_IDS_PATH.read_text(encoding='utf-8')) if EMPLOYEE_IDS_PATH.exists() else {}
zones_by_camera = {}
for zone in staff_zones:
    zones_by_camera.setdefault(str(zone['camera_id']), []).append(np.asarray(zone['polygon'], dtype=np.float32))
tracked_cameras = sorted(local_tracks['camera_id'].astype(str).unique())
for camera_id in tracked_cameras:
    if camera_id not in zones_by_camera:
        print(f'WARNING: no staff-zone polygon for {camera_id}; all tracks default to customer unless manually overridden.')
print(f'Loaded {len(local_tracks):,} local-track rows from {len(tracked_cameras)} camera(s).')


In [ ]:
# Self-contained point-in-polygon logic. It follows Notebook 02's cv2.pointPolygonTest rule,
# but uses camera pixels because staff_zones.json is intentionally camera-local.
def inside_any_staff_zone(foot_x, foot_y, polygons):
    if not polygons or not np.isfinite([foot_x, foot_y]).all():
        return False
    point = (float(foot_x), float(foot_y))
    return any(cv2.pointPolygonTest(polygon, point, False) >= 0 for polygon in polygons)

STAFF_MIN_DWELL_SECONDS = 60.0  # Change this value to tune the worker threshold.
MAX_DWELL_GAP_SECONDS = 1.0  # Never count a long tracking gap as staff time.
role_rows = []
for (camera_id, local_track_id), track in local_tracks.groupby(['camera_id', 'local_track_id'], sort=True):
    camera_id, local_track_id = str(camera_id), int(local_track_id)
    track = track.sort_values(['timestamp_sec', 'frame_index']).reset_index(drop=True)
    frames_seen = int(track['frame_index'].nunique())
    polygons = zones_by_camera.get(camera_id, [])
    in_staff_zone = track.apply(lambda row: inside_any_staff_zone(row.foot_x, row.foot_y, polygons), axis=1)
    timestamps = pd.to_numeric(track['timestamp_sec'], errors='coerce')
    elapsed_seconds = (timestamps.shift(-1) - timestamps).clip(lower=0.0, upper=MAX_DWELL_GAP_SECONDS)
    staff_zone_seconds = float(elapsed_seconds.where(in_staff_zone, 0.0).sum())
    manual_ids = {int(value) for value in employee_local_ids.get(camera_id, [])}
    if local_track_id in manual_ids:
        role, decision_source = 'staff', 'manual_override'
    elif staff_zone_seconds >= STAFF_MIN_DWELL_SECONDS:
        role, decision_source = 'staff', 'auto'
    else:
        role, decision_source = 'customer', 'auto'
    role_rows.append({'camera_id': camera_id, 'local_track_id': local_track_id, 'frames_seen': frames_seen, 'staff_zone_seconds': round(staff_zone_seconds, 3), 'role': role, 'decision_source': decision_source})

track_roles = pd.DataFrame(role_rows).sort_values(['camera_id', 'local_track_id']).reset_index(drop=True)
print(f"Role decisions: {len(track_roles):,}; staff={int((track_roles.role == 'staff').sum()):,}; customer={int((track_roles.role == 'customer').sum()):,}")
track_roles.head()


In [ ]:
TRACK_ROLES_PATH = TABLES_DIR / 'track_roles.csv'
track_roles.to_csv(TRACK_ROLES_PATH, index=False)
print(f'Saved: {TRACK_ROLES_PATH.relative_to(PROJECT_ROOT)}')


In [ ]:
def resolve_camera_video(camera_id):
    matches = [path for path in RAW_DIR.rglob('*') if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS and path.stem == camera_id]
    if not matches:
        raise FileNotFoundError(f'Raw video for {camera_id} was not found under {RAW_DIR}.')
    if len(matches) > 1:
        raise RuntimeError(f'More than one raw video matches {camera_id}: {matches}')
    return matches[0]

role_by_track = {(str(row.camera_id), int(row.local_track_id)): row.role for row in track_roles.itertuples(index=False)}
has_interpolation_metadata = 'is_interpolated' in local_tracks.columns
if not has_interpolation_metadata:
    print('INFO: local_tracks.csv has no is_interpolated column; it stores real detections only, so this replay draws real foot points only.')
for camera_id, camera_tracks in local_tracks.groupby('camera_id', sort=True):
    camera_id = str(camera_id)
    source_path = resolve_camera_video(camera_id)
    frame_rows = {int(frame_index): group for frame_index, group in camera_tracks.groupby('frame_index', sort=True)}
    capture = cv2.VideoCapture(str(source_path))
    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
    writer = None
    frame_index = 0
    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if writer is None:
                height, width = frame.shape[:2]
                output_path = VIDEOS_DIR / f'roles_{camera_id}.mp4'
                writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
                if not writer.isOpened():
                    raise RuntimeError(f'Could not create {output_path}')
            for row in frame_rows.get(frame_index, pd.DataFrame()).itertuples(index=False):
                role = role_by_track[(camera_id, int(row.local_track_id))]
                color = (0, 0, 255) if role == 'staff' else (0, 255, 0)
                point = (int(round(row.foot_x)), int(round(row.foot_y)))
                interpolated = bool(getattr(row, 'is_interpolated', False))
                cv2.circle(frame, point, 9, color, 2 if interpolated else -1, cv2.LINE_AA)
                cv2.circle(frame, point, 11, (255, 255, 255), 1, cv2.LINE_AA)
                cv2.putText(frame, f'LID {int(row.local_track_id)}', (point[0] + 9, max(20, point[1] - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
            writer.write(frame)
            frame_index += 1
    finally:
        capture.release()
        if writer is not None:
            writer.release()
    print(f'Saved {frame_index:,} replayed frames: {output_path.relative_to(PROJECT_ROOT)}')


In [ ]:
summary = (track_roles.groupby('camera_id', as_index=False).agg(staff_tracks=('role', lambda values: int((values == 'staff').sum())), customer_tracks=('role', lambda values: int((values == 'customer').sum()))).sort_values('camera_id'))
SUMMARY_PATH = TABLES_DIR / 'staff_customer_summary.csv'
summary.to_csv(SUMMARY_PATH, index=False)
print(f'Saved: {SUMMARY_PATH.relative_to(PROJECT_ROOT)}')
print(summary.to_string(index=False))
